### Transform Sprints Data

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"

In [0]:
from pyspark.sql import functions as F

#### Step 1-4 Read data, select and standardize columns

In [0]:
sprints_df = spark.table(bronze_table)

In [0]:
sprints_selected_df = sprints_df.select(
    "season",
    "round",
    "constructorId",
    "driverId",
    "date",
    "raceName",
    "grid",
    "laps",
    "number",
    "points",
    "position",
    "positionText",
    "status",
    "ingestion_timestamp",
    "source"
)

In [0]:
sprints_renamed_df = sprints_selected_df.withColumnsRenamed({
    "constructorId": "constructor_id",
    "driverId": "driver_id",
    "raceName": "race_name",
    "date": "race_date",
    "grid": "grid_position",
    "laps": "completed_laps",
    "number": "car_number",
    "position": "final_position",
    "positionText": "final_position_text"
})

#### Step 5 & 6 - Handling NULL PK's and Duplicate records

In [0]:
sprints_valid_df = sprints_renamed_df.filter(
    F.col("constructor_id").isNotNull() &
    F.col("driver_id").isNotNull() & 
    F.col("season").isNotNull() &
    F.col("round").isNotNull()
)

In [0]:
sprints_distinct_df = sprints_valid_df.dropDuplicates(["constructor_id", "driver_id", "season", "round"])

#### Step 7 - Transform values of column race_name to Title Case

In [0]:
sprints_final_df = sprints_distinct_df.withColumn("race_name", F.initcap(F.col("race_name")))

#### Step 8 - Write the data into silver sprints table

In [0]:
(
    sprints_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)